In [15]:
import pandas as pd
import matplotlib as plt
import numpy as np

In [16]:
df2 = pd.read_csv('ICSA.csv')
df2['observation_date'] = pd.to_datetime(df2['observation_date'])
df2_monthly = df2.set_index('observation_date').resample('ME').mean()
print(df2_monthly.head())
df2_monthly.to_csv("ICSA-monthly.csv")

                      ICSA
observation_date          
1967-01-31        209000.0
1967-02-28        229000.0
1967-03-31        260750.0
1967-04-30        263000.0
1967-05-31        235750.0


In [17]:
df2 = pd.read_csv('CCSA.csv')
df2['observation_date'] = pd.to_datetime(df2['observation_date'])
df2_monthly = df2.set_index('observation_date').resample('ME').mean()
df2_monthly.head()
df2_monthly.to_csv("CCSA-monthly.csv")

In [26]:
def read_all_then_join(datalist):
    data = pd.read_csv(datalist[0])

    for index, _ in enumerate(datalist):
        if index > 0:
            df = pd.read_csv(datalist[index])
            df["observation_date"] = pd.to_datetime(df["observation_date"])
            df["observation_date"] = df["observation_date"].dt.to_period("M")
            data = pd.merge(data, df, on = "observation_date", how = "inner")
        else:
            data["observation_date"] = pd.to_datetime(data["observation_date"])
            data["observation_date"] = data["observation_date"].dt.to_period("M")
            print(data.head())
    return data


data = read_all_then_join(["ICSA-monthly.csv", "JTSJOL.csv","PAYEMS.csv","UMCSENT.csv","UNRATE.csv", "JTSLDL.csv", "CCSA-monthly.csv"])
data.tail()

  observation_date      ICSA
0          1967-01  209000.0
1          1967-02  229000.0
2          1967-03  260750.0
3          1967-04  263000.0
4          1967-05  235750.0


,observation_date,ICSA,JTSJOL,PAYEMS,UMCSENT,UNRATE,JTSLDL,CCSA
290,2025-02,226000.0,7480,159155,64.7,4.1,1780,1859750.0
291,2025-03,223200.0,7200,159275,57.0,4.2,1590,1863200.0
292,2025-04,226000.0,7395,159433,52.2,4.2,1789,1872750.0
293,2025-05,234000.0,7712,159452,52.2,4.2,1611,1906000.0
294,2025-06,241250.0,7437,159466,60.7,4.1,1604,1952750.0


In [27]:
data.index = data['observation_date']
data.drop('observation_date', axis = 1, inplace = True)
data.head()

,ICSA,JTSJOL,PAYEMS,UMCSENT,UNRATE,JTSLDL,CCSA
observation_date,,,,,,,
2000-12,346000.0,5088,132716,98.4,3.9,2018,2306400.0
2001-01,340000.0,5234,132703,94.7,4.2,2220,2395750.0
2001-02,371250.0,5097,132788,90.6,4.2,1855,2486500.0
2001-03,387200.0,4762,132751,91.5,4.3,2133,2585400.0
2001-04,396750.0,4615,132457,88.4,4.4,1883,2697250.0


In [31]:
#incorrect assignment of X features and y labels. X features must be a list of lags that correspond with the appropriate label. e.g. X = [2000-1 -> 2000-6] y =  unrate for 2000-7. you must append y to X and increment the starting index of X by 1 to create the next training and label pair. e.g. X = 2000-2 -> 2000-7 Y = unrate for 2000-8.
#Input (X) = [rows 1–24 of unemployment, CPI, GDP, interest_rate]
#Label (y) = [row 25 unemployment]

X = data.loc["2000-12":"2020-1"].drop(["UNRATE"], axis=1)
y = data["UNRATE"]

In [9]:
data.head()

,Unnamed: 0_x,observation_date,ICSA,JTSJOL,PAYEMS,UMCSENT,UNRATE,JTSLDL,Unnamed: 0_y,CCSA
0,407,975628800,346000.0,5088,132716,98.4,3.9,2018,407,2306400.0
1,408,978307200,340000.0,5234,132703,94.7,4.2,2220,408,2395750.0
2,409,980985600,371250.0,5097,132788,90.6,4.2,1855,409,2486500.0
3,410,983404800,387200.0,4762,132751,91.5,4.3,2133,410,2585400.0
4,411,986083200,396750.0,4615,132457,88.4,4.4,1883,411,2697250.0


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
model = Sequential([
    LSTM(64, activation="tanh", input_shape=(10, X.shape[1])),
    Dense(1)  # predict CES
])

model.compile(optimizer="adam", loss="mse")
model.summary()
history = model.fit(X, y, epochs=50, batch_size=16, validation_split=0.2, verbose=1)



I0000 00:00:1756775118.130625   16924 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1756775119.604827   16924 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1756775119.605070   16924 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1756775119.624115   16924 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,753 (73.25 KB)

 Trainable params: 18,753 (73.25 KB)

 Non-trainable params: 0 (0.00 B)

ValueError: Invalid dtype: period[M]